<div style="background-color:#1B2A4A; padding:28px 20px; border-radius:8px; border-bottom:4px solid #C9A84C;">
  <p style="font-family:Times New Roman; text-align:center; font-size:32px; font-weight:bold; color:white; margin:0;">
    Data Extraction from JSON to CSV
  </p>
  <p style="font-family:Times New Roman; text-align:center; font-size:13px; color:#AABBCC; margin-top:8px;">
    PADS Parkinsons Disease Smartwatch Dataset &nbsp;|&nbsp; Team: DataDetectives 3 &nbsp;|&nbsp; Sprint I &nbsp;|&nbsp; March 2026
  </p>
</div>

<br>

<p style="font-family:Times New Roman; text-align:center; font-size:15px; color:#333;">
This notebook extracts raw JSON source files from the PADS dataset and exports structured CSV files for downstream SQL cleaning and Power BI analysis.
</p>

---
## Step 1 — Library Imports

Import all required Python libraries used throughout this notebook.

In [1]:
import os
import pandas as pd
import json
import numpy as np

---
## Step 2 &nbsp;—&nbsp; Base Folder Setup

Verify the current working directory and confirm the dataset folder is placed correctly.

> **Before running the cells below**, place the `pads-parkinsons-disease-smartwatch-dataset` folder inside the directory printed by this cell.

In [9]:
#First Check your base folder path by running this command. 
base_folder_path=os.getcwd()
print(base_folder_path)
#Please keep your "pads-parkinsons-disease-smartwatch-dataset" folder inside your base folder path.
#If you do this step before running below code you will not get folder path error.

C:\Users\Jenifer\Downloads


---
## Step 3 &nbsp;—&nbsp; Extract Patients

Reads every `.json` file inside the `patients/` folder. Each file represents one subject. The data is loaded into a DataFrame and exported as `Patients.csv`.

In [3]:
# Specify the folder path
folder_path = os.path.join(os.getcwd(), "pads-parkinsons-disease-smartwatch-dataset", "patients")
print(folder_path)
# List to store all records
all_records = []

# Loop through each file in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith('.json'):  # Ensure it's a JSON file
        file_path = os.path.join(folder_path, file_name)
        
        # Open and load JSON data
        with open(file_path, 'r') as f:
            data = json.load(f)
            
            # Append data to the list (assumes data is a dictionary)
            all_records.append(data)

# Convert the list of dictionaries to a DataFrame
df = pd.DataFrame(all_records)

# Specify the output CSV file name
output_csv_path = os.path.join(os.getcwd(), "pads-parkinsons-disease-smartwatch-dataset", "Patients.csv")

# Export the DataFrame to CSV
df.to_csv(output_csv_path, index=False)

print(f"CSV file has been saved at {output_csv_path}")

C:\Users\Jenifer\Downloads\pads-parkinsons-disease-smartwatch-dataset\patients
CSV file has been saved at C:\Users\Jenifer\Downloads\pads-parkinsons-disease-smartwatch-dataset\Patients.csv


In [4]:
#Checking the dataframe for Patient records
df.head()

,resource_type,id,study_id,condition,disease_comment,age_at_diagnosis,age,height,weight,gender,handedness,appearance_in_kinship,appearance_in_first_grade_kinship,effect_of_alcohol_on_tremor
0,patient,001,PADS,Healthy,-,56,56,173,78,male,right,True,True,Unknown
1,patient,002,PADS,Other Movement Disorders,Left-Sided resting tremor and hypokinesia with...,69,81,193,104,male,right,False,None,No effect
2,patient,003,PADS,Healthy,-,45,45,170,78,female,right,False,None,Unknown
3,patient,004,PADS,Parkinson's,IPS akinetic-rigid type,63,67,161,90,female,right,False,None,No effect
4,patient,005,PADS,Parkinson's,IPS tremordominant type,65,75,172,86,male,left,False,None,Unknown


---
## Step 4 &nbsp;—&nbsp; Extract Questionnaire Responses

Reads every `.json` file inside the `questionnaire/` folder.  
Each file is a FHIR-format questionnaire response — the nested `item` array is flattened so that every question-answer pair becomes one row.

In [5]:
folder_path = os.path.join(os.getcwd(), "pads-parkinsons-disease-smartwatch-dataset", "questionnaire")

records = [
    {
        "resource_type": d.get("resource_type"),
        "subject_id": d.get("subject_id"),
        "study_id": d.get("study_id"),
        "id": d.get("id"),
        "questionnaire_name": d.get("questionnaire_name"),
        "link_id": i.get("link_id"),
        "text": i.get("text"),
        "answer": i.get("answer")
    }
    for f in os.listdir(folder_path) if f.endswith(".json")
    for d in [json.load(open(os.path.join(folder_path, f)))]
    for i in d.get("item", [])
]
# Create the DataFrame and assign it to 'df_questions'
df_questions = pd.DataFrame(records)

# Save it to CSV
output_csv_path = os.path.join(os.getcwd(), "pads-parkinsons-disease-smartwatch-dataset", "questionnaire_responses.csv")
df_questions.to_csv(output_csv_path, index=False)

# View the results in Jupyter
print(f"CSV file has been saved")


CSV file has been saved


In [6]:
#Checking the dataframe for Questionnaire records
df_questions.head()

,resource_type,subject_id,study_id,id,questionnaire_name,link_id,text,answer
0,questionnaire_response,001,PADS,Non-motor Symptoms,NMS,01,Dribbling of saliva during the daytime,False
1,questionnaire_response,001,PADS,Non-motor Symptoms,NMS,02,Loss or change in your ability to taste or smell,False
2,questionnaire_response,001,PADS,Non-motor Symptoms,NMS,03,Difficulty swallowing food or drink or problem...,False
3,questionnaire_response,001,PADS,Non-motor Symptoms,NMS,04,Vomiting or feelings of sickness (nausea),False
4,questionnaire_response,001,PADS,Non-motor Symptoms,NMS,05,Constipation (less than 3 bowel movements a we...,False



---
## Step 5 — Extracting Movement and Time Sensor

Reads every `.json` file inside the `movements/` folder. Read all the timeseries and merge to the movements_timeseries_summary.csv

In [7]:
base_folder = os.path.join(os.getcwd(), "pads-parkinsons-disease-smartwatch-dataset", "movement")

rows = []

for file in os.listdir(base_folder):
    if file.endswith(".json"):
        with open(os.path.join(base_folder, file), "r", encoding="utf-8") as f:
            record = json.load(f)

        subject_id = str(record.get("subject_id"))
        study_id = record.get("study_id")
        device_id = record.get("device_id")
        record_id = record.get("id")

        for session in record.get("session", []):
            record_name = session.get("record_name")

            for rec in session.get("records", []):
                device_location = rec.get("device_location")
                file_name = rec.get("file_name")
                channels = rec.get("channels", [])

                ts_path = os.path.join(base_folder, file_name)

                if os.path.exists(ts_path):
                    df = pd.read_csv(ts_path, header=None)

                    if len(channels) == df.shape[1]:
                        df.columns = channels
                    else:
                        continue

                    rows.append({
                        "subject_id": subject_id,
                        "study_id": study_id,
                        "device_id": device_id,
                        "record_id": record_id,
                        "record_name": record_name,
                        "device_location": device_location,
                        "file_name": file_name,
                        "time_mean": df["Time"].mean(),

                        "accelerometer_x_mean": df["Accelerometer_X"].mean(),
                        "accelerometer_x_std": df["Accelerometer_X"].std(),
                        "accelerometer_x_range": df["Accelerometer_X"].max() - df["Accelerometer_X"].min(),
                        "accelerometer_x_rms": np.sqrt(np.mean(df["Accelerometer_X"]**2)),

                        "accelerometer_y_mean": df["Accelerometer_Y"].mean(),
                        "accelerometer_y_std": df["Accelerometer_Y"].std(),
                        "accelerometer_y_range": df["Accelerometer_Y"].max() - df["Accelerometer_Y"].min(),
                        "accelerometer_y_rms": np.sqrt(np.mean(df["Accelerometer_Y"]**2)),

                        "accelerometer_z_mean": df["Accelerometer_Z"].mean(),
                        "accelerometer_z_std": df["Accelerometer_Z"].std(),
                        "accelerometer_z_range": df["Accelerometer_Z"].max() - df["Accelerometer_Z"].min(),
                        "accelerometer_z_rms": np.sqrt(np.mean(df["Accelerometer_Z"]**2)),

                        "gyroscope_x_mean": df["Gyroscope_X"].mean(),
                        "gyroscope_x_std": df["Gyroscope_X"].std(),
                        "gyroscope_x_range": df["Gyroscope_X"].max() - df["Gyroscope_X"].min(),
                        "gyroscope_x_rms": np.sqrt(np.mean(df["Gyroscope_X"]**2)),

                        "gyroscope_y_mean": df["Gyroscope_Y"].mean(),
                        "gyroscope_y_std": df["Gyroscope_Y"].std(),
                        "gyroscope_y_range": df["Gyroscope_Y"].max() - df["Gyroscope_Y"].min(),
                        "gyroscope_y_rms": np.sqrt(np.mean(df["Gyroscope_Y"]**2)),

                        "gyroscope_z_mean": df["Gyroscope_Z"].mean(),
                        "gyroscope_z_std": df["Gyroscope_Z"].std(),
                        "gyroscope_z_range": df["Gyroscope_Z"].max() - df["Gyroscope_Z"].min(),
                        "gyroscope_z_rms": np.sqrt(np.mean(df["Gyroscope_Z"]**2))
                    })

movement_timeseries_summary = pd.DataFrame(rows)

output_csv_path = os.path.join(os.getcwd(), "pads-parkinsons-disease-smartwatch-dataset", "movement_timeseries_summary.csv")
movement_timeseries_summary.to_csv(
    output_csv_path,
    index=False
)
print("movement_timeseries_summary.csv created")
print(movement_timeseries_summary.shape)


movement_timeseries_summary.csv created
(10318, 32)


In [8]:
movement_timeseries_summary.head()

,subject_id,study_id,device_id,record_id,record_name,device_location,file_name,time_mean,accelerometer_x_mean,accelerometer_x_std,...,gyroscope_x_range,gyroscope_x_rms,gyroscope_y_mean,gyroscope_y_std,gyroscope_y_range,gyroscope_y_rms,gyroscope_z_mean,gyroscope_z_std,gyroscope_z_range,gyroscope_z_rms
0,001,PADS,Apple Watch Series 4,Neurological Assessment,Relaxed,LeftWrist,timeseries/001_Relaxed_LeftWrist.txt,10.229634,-0.002355,0.004686,...,0.138620,0.012584,-0.000974,0.018319,0.641788,0.018340,0.000662,0.016578,0.604364,0.016587
1,001,PADS,Apple Watch Series 4,Neurological Assessment,Relaxed,RightWrist,timeseries/001_Relaxed_RightWrist.txt,10.302822,-0.000341,0.003977,...,0.164069,0.011871,-0.001132,0.020545,0.782303,0.020571,0.000642,0.013199,0.490131,0.013211
2,001,PADS,Apple Watch Series 4,Neurological Assessment,RelaxedTask,LeftWrist,timeseries/001_RelaxedTask_LeftWrist.txt,10.229743,-0.001586,0.006425,...,0.635984,0.046154,0.003498,0.031669,0.606001,0.031854,0.001198,0.021971,0.382449,0.021998
3,001,PADS,Apple Watch Series 4,Neurological Assessment,RelaxedTask,RightWrist,timeseries/001_RelaxedTask_RightWrist.txt,10.302652,0.000329,0.005475,...,0.320575,0.023742,0.000605,0.020137,0.720006,0.020141,0.001691,0.017907,0.621846,0.017982
4,001,PADS,Apple Watch Series 4,Neurological Assessment,StretchHold,LeftWrist,timeseries/001_StretchHold_LeftWrist.txt,5.112355,-0.001594,0.005824,...,0.116842,0.013644,0.002526,0.030825,0.761030,0.030913,-0.000232,0.019985,0.461509,0.019976


---
## Extraction Summary

All three datasets have been successfully extracted from raw JSON and saved as CSV files ready for import into PostgreSQL for cleaning.

| CSV File | Description |
|----------|-------------|
| `Patients.csv` | Patient demographics — one row per subject 
| `questionnaire_responses.csv` | NMS question-answer pairs — one row per question 
| `movement_timeseries_summary.csv` | Sensor features per recording window 

**Next Step →** Import CSVs into PostgreSQL and run the cleaning pipeline (Sprint I — SQL Cleaning).